In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider
from IPython.display import display, clear_output

In [2]:
def rosenbrock(x):
    return 100.0 * (x[1] - x[0]**2.0)**2.0 + (1 - x[0])**2.0

CASE 1: INACTIVE CONSTRAINT

In [ ]:
c = 2.0
def constraint(x):
    return 2.25*x[0] - x[1] + c

def create_plot(x1_val, x2_val, points=None, point_values=None, show_annotations=True):
    # Create the plot but don't clear the output
    plt.figure(figsize=(10, 8))
    
    x = np.linspace(-1.5, 1.5, 100)
    y = np.linspace(-3.0, 5.0, 100)
    X, Y = np.meshgrid(x, y)
    Z = np.zeros_like(X)
    C = np.zeros_like(X)  # Array to store constraint values

    for i in range(len(x)):
        for j in range(len(y)):
            Z[j, i] = rosenbrock([X[j, i], Y[j, i]])
            C[j, i] = -2.25*X[j, i] + Y[j, i]  # Constraint values

    # Plot contours of the Rosenbrock function
    levels = np.logspace(-1, 3, 20)
    contour = plt.contourf(X, Y, Z, levels=levels, alpha=0.6, cmap='viridis')
    plt.colorbar(contour, label='Rosenbrock function value')

    # Plot contours of the constraint function
    constraint_levels = np.linspace(-4, 4, 9)
    constraint_contour = plt.contour(X, Y, C, levels=constraint_levels, colors='red', linestyles='dashed', linewidths=1)
    plt.clabel(constraint_contour, inline=True, fontsize=8, fmt='%1.1f')

    # Plot the constraint line: -2.25*x[0] + x[1] = c
    constraint_x = np.linspace(-1.5, 1.5, 100)
    constraint_y = 2.25 * constraint_x + c
    plt.plot(constraint_x, constraint_y, 'r-', linewidth=2, label='Constraint: -2.25*x1 + x2 = c')

    # Shade the infeasible region (where -2.25*x[0] + x[1] > 2)
    plt.fill_between(constraint_x, constraint_y, 5, alpha=0.2, color='red', label='Infeasible region')

    # Mark the current point
    current_point = np.array([x1_val, x2_val])
    plt.plot(current_point[0], current_point[1], 'bo', markersize=8, label=f'Initial point: ({x1_val:.4f}, {x2_val:.4f})')
    
    solution_point = np.array([1.0, 1.0])
    plt.plot(solution_point[0], solution_point[1], 'bx', markersize=12, label='Solution')
    
    # Plot additional points if provided
    if points is not None and len(points) > 0:
        points_array = np.array(points)

        if len(points) > 0:
            first_point = points[0]
            plt.plot([x1_val, first_point[0]], [x2_val, first_point[1]], 'g--', linewidth=1)

        plt.plot(points_array[:, 0], points_array[:, 1], 'g--', linewidth=1)
        plt.plot(points_array[:, 0], points_array[:, 1], 'go', markersize=4, label='Iterated points')
        
        # Add minimal annotations (just point numbers) if desired
        if show_annotations:
            for i, point in enumerate(points):
                plt.annotate(
                    f"{i+1}",
                    (point[0], point[1]),
                    xytext=(5, 5),
                    textcoords='offset points',
                    fontsize=9,
                    bbox=dict(boxstyle="circle", fc="white", alpha=0.8)
                )
    
    # Calculate function value and constraint status
    f_val = rosenbrock(current_point)
    constraint_val = -2.25*current_point[0] + current_point[1]
    is_feasible = constraint_val <= -1.5
    
    # Add labels and legend
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.xlim((-1.5, 1.5))
    plt.ylim((-3.0, 5.5))
    plt.title('Interactive Constrained Rosenbrock Function (-2.25*x1 + x2 <= c)')
    plt.legend(loc='lower right')
    plt.grid(True)
    
    # Return the point data for printing below
    return {
        'current_point': (x1_val, x2_val, f_val, constraint_val, is_feasible),
        'points': points,
        'point_values': point_values
    }

# Create interactive widgets
def interactive_rosenbrock():
    points = []  # Initialize empty list for points
    point_values = []  # Store function values
    
    # Create a fixed output area for the plot and information
    plot_output = widgets.Output()
    info_output = widgets.Output()
    
    # Create widgets
    x1_slider = FloatSlider(min=-1.5, max=1.5, step=0.01, value=-1.3, description='x1:')
    x2_slider = FloatSlider(min=-3.0, max=5.0, step=0.01, value=-1.0, description='x2:')
    points_input = widgets.Textarea(
        value='',
        placeholder='Enter points as "x1,x2" pairs (one per line)\nExample:\n0.5,1.2\n-0.3,0.7',
        description='Points:',
        layout=widgets.Layout(width='300px', height='100px')
    )
    
    add_button = widgets.Button(description="Add Current Point")
    plot_points_button = widgets.Button(description="Plot Entered Points")
    clear_button = widgets.Button(description="Clear Points")
    show_annotations_checkbox = widgets.Checkbox(value=False, description='Show point numbers', disabled=False)
    
    # Add trust region widgets
    trust_center_input = widgets.Text(
        value='0.0,0.0',
        placeholder='x1,x2',
        description='Trust center:',
        layout=widgets.Layout(width='300px')
    )
    
    trust_radius_input = widgets.FloatText(
        value=0.5,
        description='Trust radius:',
        min=0.1,
        max=5.0,
        step=0.1,
        layout=widgets.Layout(width='200px')
    )
    
    show_trust_checkbox = widgets.Checkbox(
        value=False,
        description='Show trust region',
        layout=widgets.Layout(width='150px')
    )
    
    def update_plot(x1_val, x2_val):
        with plot_output:
            plot_output.clear_output(wait=True)
            result = create_plot(
                x1_val, x2_val, points, point_values, show_annotations_checkbox.value
            )
            
            # Add trust region if enabled
            if show_trust_checkbox.value:
                try:
                    # Parse trust center
                    center = list(map(float, trust_center_input.value.split(',')))
                    radius = trust_radius_input.value
                    
                    # Plot trust region circle
                    theta = np.linspace(0, 2*np.pi, 100)
                    x = center[0] + radius * np.cos(theta)
                    y = center[1] + radius * np.sin(theta)
                    plt.plot(x, y, 'b-', linewidth=2)
                    
                    # Mark the trust center
                    plt.plot(center[0], center[1], 'bx', markersize=8)
                    # plt.annotate('Trust center', 
                    #              (center[0], center[1]),
                    #              xytext=(10, 10),
                    #              textcoords='offset points',
                    #              fontsize=9)
                except Exception as e:
                    plt.figtext(0.5, 0.01, f"Error plotting trust region: {e}", 
                                ha='center', color='red')
            
            plt.show()
        
        with info_output:
            info_output.clear_output(wait=True)
            # Print current point info
            curr = result['current_point']
            print(f"Initial point: x1 = {curr[0]:.6f}, x2 = {curr[1]:.6f}")
            print(f"Objective function value: {curr[2]:.6f}")
            print(f"Constraint value: {curr[3]:.6f} {'(Feasible)' if curr[4] else '(Infeasible)'}")
            
            # Print all points info if there are any
            if points and point_values:
                print("\nIteration Progress:")
                for i, (point, value) in enumerate(zip(points, point_values)):
                    constraint_val = -2.25*point[0] + point[1]
                    is_feasible = constraint_val <= c
                    status = "FEASIBLE" if is_feasible else "INFEASIBLE"
                    print(f"Iteration {i+1}: ({point[0]:.4f}, {point[1]:.4f}) | f={value:.6f} | c={constraint_val:.6f} | {status}")
            
            print("\nSolution:")
            print("Unconstrained global minimum is at (1.0, 1.0) with value 0.0")
    
    def on_add_clicked(b):
        nonlocal points, point_values
        new_point = [x1_slider.value, x2_slider.value]
        points.append(new_point)
        point_values.append(rosenbrock(new_point))
        update_plot(x1_slider.value, x2_slider.value)
    
    def on_plot_points_clicked(b):
        nonlocal points, point_values
        try:
            # Parse the text input to get points
            text_points = points_input.value.strip().split('\n')
            new_points = []
            new_values = []
            
            for point_text in text_points:
                if point_text.strip():
                    coords = list(map(float, point_text.split(',')))
                    if len(coords) == 2:
                        new_points.append(coords)
                        new_values.append(rosenbrock(coords))
            
            points = new_points
            point_values = new_values
            update_plot(x1_slider.value, x2_slider.value)
        except Exception as e:
            with info_output:
                info_output.clear_output(wait=True)
                print(f"Error parsing points: {e}")
                print("Make sure each line contains a comma-separated pair of numbers.")
    
    def on_clear_clicked(b):
        nonlocal points, point_values
        points = []
        point_values = []
        points_input.value = ''
        update_plot(x1_slider.value, x2_slider.value)
    
    add_button.on_click(on_add_clicked)
    plot_points_button.on_click(on_plot_points_clicked)
    clear_button.on_click(on_clear_clicked)
    
     # Display widgets first, then update the plot
    display(widgets.HBox([x1_slider, x2_slider]))
    display(points_input)
    display(widgets.HBox([add_button, plot_points_button, clear_button, show_annotations_checkbox]))
    display(widgets.HBox([trust_center_input, trust_radius_input, show_trust_checkbox]))
    display(plot_output)
    display(info_output)
    
    # Update plot when sliders change
    def on_value_change(change):
        update_plot(x1_slider.value, x2_slider.value)
    
    x1_slider.observe(on_value_change, names='value')
    x2_slider.observe(on_value_change, names='value')
    show_annotations_checkbox.observe(lambda change: update_plot(x1_slider.value, x2_slider.value), names='value')
    
    # Update plot when trust region settings change
    trust_center_input.observe(lambda change: update_plot(x1_slider.value, x2_slider.value), names='value')
    trust_radius_input.observe(lambda change: update_plot(x1_slider.value, x2_slider.value), names='value')
    show_trust_checkbox.observe(lambda change: update_plot(x1_slider.value, x2_slider.value), names='value')
    
    # Initial plot - after all widgets are displayed
    update_plot(x1_slider.value, x2_slider.value)

# Run the interactive visualization
interactive_rosenbrock()

Textarea(value='', description='Points:', layout=Layout(height='100px', width='300px'), placeholder='Enter poi…

Output()

Output()

CASE 2: ONE ACTIVE LINEAR CONSTRAINT

In [4]:
c2 = -1.5
def constraint2(x):
    return 2.25*x[0] - x[1]  # This is equivalent to -2.25*x[0] + x[1] <= 2

def create_plot2(x1_val, x2_val, points=None, point_values=None, show_annotations=True):
    # Create the plot but don't clear the output
    plt.figure(figsize=(10, 8))
    
    x = np.linspace(-1.5, 1.5, 100)
    y = np.linspace(-3.0, 5.0, 100)
    X, Y = np.meshgrid(x, y)
    Z = np.zeros_like(X)
    C = np.zeros_like(X)  # Array to store constraint values

    for i in range(len(x)):
        for j in range(len(y)):
            Z[j, i] = rosenbrock([X[j, i], Y[j, i]])
            C[j, i] = constraint2([X[j, i], Y[j, i]])  # Constraint values

    # Plot contours of the Rosenbrock function
    levels = np.logspace(-1, 3, 20)
    contour = plt.contourf(X, Y, Z, levels=levels, alpha=0.6, cmap='viridis')
    plt.colorbar(contour, label='Rosenbrock function value')

    # Plot contours of the constraint function
    constraint_levels = np.linspace(-4, 4, 9)
    constraint_contour = plt.contour(X, Y, C, levels=constraint_levels, colors='red', linestyles='dashed', linewidths=1)
    plt.clabel(constraint_contour, inline=True, fontsize=8, fmt='%1.1f')

    # Plot the constraint line: -2.25*x[0] + x[1] = c
    constraint_x = np.linspace(-1.5, 1.5, 100)
    constraint_y = 2.25 * constraint_x + c2
    plt.plot(constraint_x, constraint_y, 'r-', linewidth=2, label='Constraint: -2.25*x1 + x2 = c')

    # Shade the infeasible region (where -2.25*x[0] + x[1] > 2)
    plt.fill_between(constraint_x, constraint_y, 5, alpha=0.2, color='red', label='Infeasible region')

    # Mark the current point
    current_point = np.array([x1_val, x2_val])
    plt.plot(current_point[0], current_point[1], 'bo', markersize=8, label=f'Initial point: ({x1_val:.4f}, {x2_val:.4f})')
    
    solution_point = np.array([1.1224, 1.0254])
    plt.plot(solution_point[0], solution_point[1], 'bx', markersize=12, label='Solution')

    # Plot additional points if provided
    if points is not None and len(points) > 0:
        points_array = np.array(points)

        if len(points) > 0:
            first_point = points[0]
            plt.plot([x1_val, first_point[0]], [x2_val, first_point[1]], 'g--', linewidth=1)

        plt.plot(points_array[:, 0], points_array[:, 1], 'g--', linewidth=1)
        plt.plot(points_array[:, 0], points_array[:, 1], 'go', markersize=4, label='Iterated points')
        
        # Add minimal annotations (just point numbers) if desired
        if show_annotations:
            for i, point in enumerate(points):
                plt.annotate(
                    f"{i+1}",
                    (point[0], point[1]),
                    xytext=(5, 5),
                    textcoords='offset points',
                    fontsize=9,
                    bbox=dict(boxstyle="circle", fc="white", alpha=0.8)
                )
    
    # Calculate function value and constraint status
    f_val = rosenbrock(current_point)
    constraint_val = -2.25*current_point[0] + current_point[1]
    is_feasible = constraint_val <= -1.5
    
    # Add labels and legend
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.xlim((-1.5, 1.5))
    plt.ylim((-3.0, 5.5))
    plt.title('Interactive Constrained Rosenbrock Function (-2.25*x1 + x2 <= c)')
    plt.legend(loc='lower right')
    plt.grid(True)
    
    # Return the point data for printing below
    return {
        'current_point': (x1_val, x2_val, f_val, constraint_val, is_feasible),
        'points': points,
        'point_values': point_values
    }

# Create interactive widgets
def interactive_rosenbrock2():
    points = []  # Initialize empty list for points
    point_values = []  # Store function values
    
    # Create a fixed output area for the plot and information
    plot_output = widgets.Output()
    info_output = widgets.Output()
    
    # Create widgets
    x1_slider = FloatSlider(min=-1.5, max=1.5, step=0.01, value=0.1, description='x1:')
    x2_slider = FloatSlider(min=-3.0, max=5.0, step=0.01, value=-1.6, description='x2:')
    points_input = widgets.Textarea(
        value='',
        placeholder='Enter points as "x1,x2" pairs (one per line)\nExample:\n0.5,1.2\n-0.3,0.7',
        description='Points:',
        layout=widgets.Layout(width='300px', height='100px')
    )
    
    add_button = widgets.Button(description="Add Current Point")
    plot_points_button = widgets.Button(description="Plot Entered Points")
    clear_button = widgets.Button(description="Clear Points")
    show_annotations_checkbox = widgets.Checkbox(value=False, description='Show point numbers', disabled=False)
    
    # Add trust region widgets
    trust_center_input = widgets.Text(
        value='0.0,0.0',
        placeholder='x1,x2',
        description='Trust center:',
        layout=widgets.Layout(width='300px')
    )
    
    trust_radius_input = widgets.FloatText(
        value=0.5,
        description='Trust radius:',
        min=0.1,
        max=5.0,
        step=0.1,
        layout=widgets.Layout(width='200px')
    )
    
    show_trust_checkbox = widgets.Checkbox(
        value=False,
        description='Show trust region',
        layout=widgets.Layout(width='150px')
    )
    
    def update_plot2(x1_val, x2_val):
        with plot_output:
            plot_output.clear_output(wait=True)
            result = create_plot2(
                x1_val, x2_val, points, point_values, show_annotations_checkbox.value
            )
            
            # Add trust region if enabled
            if show_trust_checkbox.value:
                try:
                    # Parse trust center
                    center = list(map(float, trust_center_input.value.split(',')))
                    radius = trust_radius_input.value
                    
                    # Plot trust region circle
                    theta = np.linspace(0, 2*np.pi, 100)
                    x = center[0] + radius * np.cos(theta)
                    y = center[1] + radius * np.sin(theta)
                    plt.plot(x, y, 'b-', linewidth=2)
                    
                    # Mark the trust center
                    plt.plot(center[0], center[1], 'bx', markersize=8)
                    # plt.annotate('Trust center', 
                    #              (center[0], center[1]),
                    #              xytext=(10, 10),
                    #              textcoords='offset points',
                    #              fontsize=9)
                except Exception as e:
                    plt.figtext(0.5, 0.01, f"Error plotting trust region: {e}", 
                                ha='center', color='red')
            
            plt.show()
        
        with info_output:
            info_output.clear_output(wait=True)
            # Print current point info
            curr = result['current_point']
            print(f"Initial point: x1 = {curr[0]:.6f}, x2 = {curr[1]:.6f}")
            print(f"Objective function value: {curr[2]:.6f}")
            print(f"Constraint value: {curr[3]:.6f} {'(Feasible)' if curr[4] else '(Infeasible)'}")
            
            # Print all points info if there are any
            if points and point_values:
                print("\nIteration Progress:")
                for i, (point, value) in enumerate(zip(points, point_values)):
                    constraint_val = -2.25*point[0] + point[1]
                    is_feasible = constraint_val <= c
                    status = "FEASIBLE" if is_feasible else "INFEASIBLE"
                    print(f"Iteration {i+1}: ({point[0]:.4f}, {point[1]:.4f}) | f={value:.6f} | c={constraint_val:.6f} | {status}")
            
            print("\nReference:")
            print("Constrained global minimum is at (1.1224, 1.0254) with value 5.5085")
    
    def on_add_clicked2(b):
        nonlocal points, point_values
        new_point = [x1_slider.value, x2_slider.value]
        points.append(new_point)
        point_values.append(rosenbrock(new_point))
        update_plot2(x1_slider.value, x2_slider.value)
    
    def on_plot_points_clicked2(b):
        nonlocal points, point_values
        try:
            # Parse the text input to get points
            text_points = points_input.value.strip().split('\n')
            new_points = []
            new_values = []
            
            for point_text in text_points:
                if point_text.strip():
                    coords = list(map(float, point_text.split(',')))
                    if len(coords) == 2:
                        new_points.append(coords)
                        new_values.append(rosenbrock(coords))
            
            points = new_points
            point_values = new_values
            update_plot2(x1_slider.value, x2_slider.value)
        except Exception as e:
            with info_output:
                info_output.clear_output(wait=True)
                print(f"Error parsing points: {e}")
                print("Make sure each line contains a comma-separated pair of numbers.")
    
    def on_clear_clicked2(b):
        nonlocal points, point_values
        points = []
        point_values = []
        points_input.value = ''
        update_plot2(x1_slider.value, x2_slider.value)
    
    add_button.on_click(on_add_clicked2)
    plot_points_button.on_click(on_plot_points_clicked2)
    clear_button.on_click(on_clear_clicked2)
    
     # Display widgets first, then update the plot
    display(widgets.HBox([x1_slider, x2_slider]))
    display(points_input)
    display(widgets.HBox([add_button, plot_points_button, clear_button, show_annotations_checkbox]))
    display(widgets.HBox([trust_center_input, trust_radius_input, show_trust_checkbox]))
    display(plot_output)
    display(info_output)
    
    # Update plot when sliders change   
    def on_value_change(change):
        update_plot2(x1_slider.value, x2_slider.value)
    
    x1_slider.observe(on_value_change, names='value')
    x2_slider.observe(on_value_change, names='value')
    show_annotations_checkbox.observe(lambda change: update_plot2(x1_slider.value, x2_slider.value), names='value')
    
    # Update plot when trust region settings change
    trust_center_input.observe(lambda change: update_plot2(x1_slider.value, x2_slider.value), names='value')
    trust_radius_input.observe(lambda change: update_plot2(x1_slider.value, x2_slider.value), names='value')
    show_trust_checkbox.observe(lambda change: update_plot2(x1_slider.value, x2_slider.value), names='value')
    
    # Initial plot - after all widgets are displayed
    update_plot2(x1_slider.value, x2_slider.value)   

# Run the interactive visualization
interactive_rosenbrock2()

Textarea(value='', description='Points:', layout=Layout(height='100px', width='300px'), placeholder='Enter poi…

Output()

Output()

CASE 3: ONE ACTIVE NON-LINEAR CONSTRAINT

In [5]:
c3 = -1.5
def constraint3(x):
    return -2.25*x[0]**2 + x[1]

def create_plot3(x1_val, x2_val, points=None, point_values=None, show_annotations=True):
    # Create the plot but don't clear the output
    plt.figure(figsize=(10, 8))
    
    x = np.linspace(-2.5, 2.5, 100)
    y = np.linspace(-3.0, 5.0, 100)
    X, Y = np.meshgrid(x, y)
    Z = np.zeros_like(X)
    C = np.zeros_like(X)  # Array to store constraint values

    for i in range(len(x)):
        for j in range(len(y)):
            Z[j, i] = rosenbrock([X[j, i], Y[j, i]])
            C[j, i] = constraint3([X[j, i], Y[j, i]])  # Constraint values

    # Plot contours of the Rosenbrock function
    levels = np.logspace(-1, 3, 20)
    contour = plt.contourf(X, Y, Z, levels=levels, alpha=0.6, cmap='viridis')
    plt.colorbar(contour, label='Rosenbrock function value')

    # Plot contours of the constraint function
    constraint_levels = np.linspace(-4, 4, 9)
    constraint_contour = plt.contour(X, Y, C, levels=constraint_levels, colors='red', linestyles='dashed', linewidths=1)
    plt.clabel(constraint_contour, inline=True, fontsize=8, fmt='%1.1f')

    
    constraint_x = np.linspace(-2.5, 2.5, 100)
    constraint_y = 2.25 * constraint_x**2 + c3
    plt.plot(constraint_x, constraint_y, 'r-', linewidth=2, label='Constraint: -2.25*x1^2 + x2 = c')

    plt.fill_between(constraint_x, constraint_y, 5, alpha=0.2, color='red', label='Infeasible region')

    # Mark the current point
    current_point = np.array([x1_val, x2_val])
    plt.plot(current_point[0], current_point[1], 'bo', markersize=8, label=f'Initial point: ({x1_val:.4f}, {x2_val:.4f})')
    
    solution_point = np.array([1.09531799296728, 1.19937338785603])
    plt.plot(solution_point[0], solution_point[1], 'bx', markersize=12, label='Solution')

    # Plot additional points if provided
    if points is not None and len(points) > 0:
        points_array = np.array(points)

        if len(points) > 0:
            first_point = points[0]
            plt.plot([x1_val, first_point[0]], [x2_val, first_point[1]], 'g--', linewidth=1)

        plt.plot(points_array[:, 0], points_array[:, 1], 'g--', linewidth=1)
        plt.plot(points_array[:, 0], points_array[:, 1], 'go', markersize=4, label='Iterated points')
        
        # Add minimal annotations (just point numbers) if desired
        if show_annotations:
            for i, point in enumerate(points):
                plt.annotate(
                    f"{i+1}",
                    (point[0], point[1]),
                    xytext=(5, 5),
                    textcoords='offset points',
                    fontsize=9,
                    bbox=dict(boxstyle="circle", fc="white", alpha=0.8)
                )
    
    # Calculate function value and constraint status
    f_val = rosenbrock(current_point)
    constraint_val = -2.25*current_point[0]**2 + current_point[1]
    is_feasible = constraint_val <= c3
    
    # Add labels and legend
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.xlim((-2.5, 2.5))
    plt.ylim((-3.0, 5.5))
    plt.title('Interactive Constrained Rosenbrock Function (2.25*x1^2 - x2 <= c)')
    plt.legend(loc='lower left')
    plt.grid(True)
    
    # Return the point data for printing below
    return {
        'current_point': (x1_val, x2_val, f_val, constraint_val, is_feasible),
        'points': points,
        'point_values': point_values
    }

# Create interactive widgets
def interactive_rosenbrock3():
    points = []  # Initialize empty list for points
    point_values = []  # Store function values
    
    # Create a fixed output area for the plot and information
    plot_output = widgets.Output()
    info_output = widgets.Output()
    
    # Create widgets
    x1_slider = FloatSlider(min=-1.5, max=1.5, step=0.01, value=0.1, description='x1:')
    x2_slider = FloatSlider(min=-3.0, max=5.0, step=0.01, value=-1.6, description='x2:')
    points_input = widgets.Textarea(
        value='',
        placeholder='Enter points as "x1,x2" pairs (one per line)\nExample:\n0.5,1.2\n-0.3,0.7',
        description='Points:',
        layout=widgets.Layout(width='300px', height='100px')
    )
    
    add_button = widgets.Button(description="Add Current Point")
    plot_points_button = widgets.Button(description="Plot Entered Points")
    clear_button = widgets.Button(description="Clear Points")
    show_annotations_checkbox = widgets.Checkbox(value=False, description='Show point numbers', disabled=False)
    
    # Add trust region widgets
    trust_center_input = widgets.Text(
        value='0.0,0.0',
        placeholder='x1,x2',
        description='Trust center:',
        layout=widgets.Layout(width='300px')
    )
    
    trust_radius_input = widgets.FloatText(
        value=0.5,
        description='Trust radius:',
        min=0.1,
        max=5.0,
        step=0.1,
        layout=widgets.Layout(width='200px')
    )
    
    show_trust_checkbox = widgets.Checkbox(
        value=False,
        description='Show trust region',
        layout=widgets.Layout(width='150px')
    )
    
    def update_plot3(x1_val, x2_val):
        with plot_output:
            plot_output.clear_output(wait=True)
            result = create_plot3(
                x1_val, x2_val, points, point_values, show_annotations_checkbox.value
            )
            
            # Add trust region if enabled
            if show_trust_checkbox.value:
                try:
                    # Parse trust center
                    center = list(map(float, trust_center_input.value.split(',')))
                    radius = trust_radius_input.value
                    
                    # Plot trust region circle
                    theta = np.linspace(0, 2*np.pi, 100)
                    x = center[0] + radius * np.cos(theta)
                    y = center[1] + radius * np.sin(theta)
                    plt.plot(x, y, 'b-', linewidth=2)
                    
                    # Mark the trust center
                    plt.plot(center[0], center[1], 'bx', markersize=8)
                    # plt.annotate('Trust center', 
                    #              (center[0], center[1]),
                    #              xytext=(10, 10),
                    #              textcoords='offset points',
                    #              fontsize=9)
                except Exception as e:
                    plt.figtext(0.5, 0.01, f"Error plotting trust region: {e}", 
                                ha='center', color='red')
            
            plt.show()
        
        with info_output:
            info_output.clear_output(wait=True)
            # Print current point info
            curr = result['current_point']
            print(f"Initial point: x1 = {curr[0]:.6f}, x2 = {curr[1]:.6f}")
            print(f"Objective function value: {curr[2]:.6f}")
            print(f"Constraint value: {curr[3]:.6f} {'(Feasible)' if curr[4] else '(Infeasible)'}")
            
            # Print all points info if there are any
            if points and point_values:
                print("\nIteration Progress:")
                for i, (point, value) in enumerate(zip(points, point_values)):
                    constraint_val = constraint3(point)
                    is_feasible = constraint_val <= c3
                    status = "FEASIBLE" if is_feasible else "INFEASIBLE"
                    print(f"Iteration {i+1}: ({point[0]:.4f}, {point[1]:.4f}) | f={value:.6f} | c={constraint_val:.6f} | {status}")
            
            print("\nReference:")
            print("Constrained global minimum is at (1.0953, 1.199373) with value 0.0")
    
    def on_add_clicked3(b):
        nonlocal points, point_values
        new_point = [x1_slider.value, x2_slider.value]
        points.append(new_point)
        point_values.append(rosenbrock(new_point))
        update_plot3(x1_slider.value, x2_slider.value)
    
    def on_plot_points_clicked3(b):
        nonlocal points, point_values
        try:
            # Parse the text input to get points
            text_points = points_input.value.strip().split('\n')
            new_points = []
            new_values = []
            
            for point_text in text_points:
                if point_text.strip():
                    coords = list(map(float, point_text.split(',')))
                    if len(coords) == 2:
                        new_points.append(coords)
                        new_values.append(rosenbrock(coords))
            
            points = new_points
            point_values = new_values
            update_plot3(x1_slider.value, x2_slider.value)
        except Exception as e:
            with info_output:
                info_output.clear_output(wait=True)
                print(f"Error parsing points: {e}")
                print("Make sure each line contains a comma-separated pair of numbers.")
    
    def on_clear_clicked3(b):
        nonlocal points, point_values
        points = []
        point_values = []
        points_input.value = ''
        update_plot3(x1_slider.value, x2_slider.value)
    
    add_button.on_click(on_add_clicked3)
    plot_points_button.on_click(on_plot_points_clicked3)
    clear_button.on_click(on_clear_clicked3)
    
     # Display widgets first, then update the plot
    display(widgets.HBox([x1_slider, x2_slider]))
    display(points_input)
    display(widgets.HBox([add_button, plot_points_button, clear_button, show_annotations_checkbox]))
    display(widgets.HBox([trust_center_input, trust_radius_input, show_trust_checkbox]))
    display(plot_output)
    display(info_output)
    
    # Update plot when sliders change
    def on_value_change(change):
        update_plot3(x1_slider.value, x2_slider.value)
    
    x1_slider.observe(on_value_change, names='value')
    x2_slider.observe(on_value_change, names='value')
    show_annotations_checkbox.observe(lambda change: update_plot3(x1_slider.value, x2_slider.value), names='value')
    
    # Update plot when trust region settings change
    trust_center_input.observe(lambda change: update_plot3(x1_slider.value, x2_slider.value), names='value')
    trust_radius_input.observe(lambda change: update_plot3(x1_slider.value, x2_slider.value), names='value')
    show_trust_checkbox.observe(lambda change: update_plot3(x1_slider.value, x2_slider.value), names='value')
    
    # Initial plot - after all widgets are displayed
    update_plot3(x1_slider.value, x2_slider.value)

# Run the interactive visualization
interactive_rosenbrock3()

Textarea(value='', description='Points:', layout=Layout(height='100px', width='300px'), placeholder='Enter poi…

Output()

Output()

CASE 3: TWO CONSTRAINTS WITH ONE ACTIVE

In [6]:
c1 = -0.5
c2 = 0.0 

def constraint1(x):
    return -2.25*x[0] + x[1] - c1

def constraint2(x):
    return x[0] + 1.5*x[1] - c2

def create_plot(x1_val, x2_val, points=None, point_values=None, show_annotations=True):
    # Create the plot but don't clear the output
    plt.figure(figsize=(10, 8))
    
    x = np.linspace(-1.5, 1.5, 100)
    y = np.linspace(-3.0, 5.0, 100)
    X, Y = np.meshgrid(x, y)
    Z = np.zeros_like(X)
    C1 = np.zeros_like(X)  # Array to store first constraint values
    C2 = np.zeros_like(X)  # Array to store second constraint values

    for i in range(len(x)):
        for j in range(len(y)):
            Z[j, i] = rosenbrock([X[j, i], Y[j, i]])
            C1[j, i] = -2.25*X[j, i] + Y[j, i]  # First constraint values
            C2[j, i] = X[j, i] + 1.5*Y[j, i]    # Second constraint values

    # Plot contours of the Rosenbrock function
    levels = np.logspace(-1, 3, 20)
    contour = plt.contourf(X, Y, Z, levels=levels, alpha=0.6, cmap='viridis')
    plt.colorbar(contour, label='Rosenbrock function value')

    # Plot contours of the first constraint function
    constraint_levels = np.linspace(-4, 4, 9)
    constraint_contour1 = plt.contour(X, Y, C1, levels=constraint_levels, colors='red', linestyles='dashed', linewidths=1)
    plt.clabel(constraint_contour1, inline=True, fontsize=8, fmt='%1.1f')
    
    # Plot contours of the second constraint function
    constraint_contour2 = plt.contour(X, Y, C2, levels=constraint_levels, colors='indigo', linestyles='dashed', linewidths=1)
    plt.clabel(constraint_contour2, inline=True, fontsize=8, fmt='%1.1f')

    # Plot the first constraint line: -2.25*x[0] + x[1] = c1
    constraint_x = np.linspace(-1.5, 1.5, 100)
    constraint1_y = 2.25 * constraint_x + c1
    plt.plot(constraint_x, constraint1_y, 'r-', linewidth=2, label='Constraint 1: -2.25*x1 + x2 = c1')

    # Plot the second constraint line: x[0] + 1.5*x[1] = c2
    constraint2_y = (-constraint_x - c2) / 1.5
    plt.plot(constraint_x, constraint2_y, ls = '-', color = 'indigo', linewidth=2, label='Constraint 2: x1 + 1.5*x2 = c2')

    # Shade the infeasible regions
    plt.fill_between(constraint_x, constraint1_y, 5, alpha=0.2, color='red')
    plt.fill_between(constraint_x, constraint2_y, 5, alpha=0.2, color='indigo')

    # Mark the current point
    current_point = np.array([x1_val, x2_val])
    plt.plot(current_point[0], current_point[1], 'bo', markersize=8, label=f'Initial point: ({x1_val:.4f}, {x2_val:.4f})')
    
    # Mark the solution point (this might need to be updated based on the new constraints)
    solution_point = np.array([0.17143, -0.11429])
    plt.plot(solution_point[0], solution_point[1], 'bx', markersize=12, label='Solution')

    # Plot additional points if provided
    if points is not None and len(points) > 0:
        points_array = np.array(points)

        if len(points) > 0:
            first_point = points[0]
            plt.plot([x1_val, first_point[0]], [x2_val, first_point[1]], 'g--', linewidth=1)

        plt.plot(points_array[:, 0], points_array[:, 1], 'g--', linewidth=1)
        plt.plot(points_array[:, 0], points_array[:, 1], 'go', markersize=4, label='Iterated points')
        
        # Add minimal annotations (just point numbers) if desired
        if show_annotations:
            for i, point in enumerate(points):
                plt.annotate(
                    f"{i+1}",
                    (point[0], point[1]),
                    xytext=(5, 5),
                    textcoords='offset points',
                    fontsize=9,
                    bbox=dict(boxstyle="circle", fc="white", alpha=0.8)
                )
    
    # Calculate function value and constraint status
    f_val = rosenbrock(current_point)
    constraint1_val = -2.25*current_point[0] + current_point[1]
    constraint2_val = current_point[0] + 1.5*current_point[1]
    is_feasible1 = constraint1_val <= c1
    is_feasible2 = constraint2_val <= c2
    is_feasible = is_feasible1 and is_feasible2
    
    # Add labels and legend
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.xlim((-1.5, 1.5))
    plt.ylim((-3.0, 5.5))
    plt.title('Interactive Constrained Rosenbrock Function with Two Constraints')
    plt.legend(loc='lower right')
    plt.grid(True)
    
    # Return the point data for printing below
    return {
        'current_point': (x1_val, x2_val, f_val, constraint1_val, constraint2_val, is_feasible1, is_feasible2),
        'points': points,
        'point_values': point_values
    }

# Create interactive widgets
def interactive_rosenbrock():
    points = []  # Initialize empty list for points
    point_values = []  # Store function values
    
    # Create a fixed output area for the plot and information
    plot_output = widgets.Output()
    info_output = widgets.Output()
    
    # Create widgets
    x1_slider = FloatSlider(min=-1.5, max=1.5, step=0.01, value=0.1, description='x1:')
    x2_slider = FloatSlider(min=-3.0, max=5.0, step=0.01, value=-1.6, description='x2:')
    points_input = widgets.Textarea(
        value='',
        placeholder='Enter points as "x1,x2" pairs (one per line)\nExample:\n0.5,1.2\n-0.3,0.7',
        description='Points:',
        layout=widgets.Layout(width='300px', height='100px')
    )
    
    add_button = widgets.Button(description="Add Current Point")
    plot_points_button = widgets.Button(description="Plot Entered Points")
    clear_button = widgets.Button(description="Clear Points")
    show_annotations_checkbox = widgets.Checkbox(value=False, description='Show point numbers', disabled=False)
    
    # Add trust region widgets
    trust_center_input = widgets.Text(
        value='0.0,0.0',
        placeholder='x1,x2',
        description='Trust center:',
        layout=widgets.Layout(width='300px')
    )
    
    trust_radius_input = widgets.FloatText(
        value=0.5,
        description='Trust radius:',
        min=0.1,
        max=5.0,
        step=0.1,
        layout=widgets.Layout(width='200px')
    )
    
    show_trust_checkbox = widgets.Checkbox(
        value=False,
        description='Show trust region',
        layout=widgets.Layout(width='150px')
    )
    
    def update_plot(x1_val, x2_val):
        with plot_output:
            plot_output.clear_output(wait=True)
            result = create_plot(
                x1_val, x2_val, points, point_values, show_annotations_checkbox.value
            )
            
            # Add trust region if enabled
            if show_trust_checkbox.value:
                try:
                    # Parse trust center
                    center = list(map(float, trust_center_input.value.split(',')))
                    radius = trust_radius_input.value
                    
                    # Plot trust region circle
                    theta = np.linspace(0, 2*np.pi, 100)
                    x = center[0] + radius * np.cos(theta)
                    y = center[1] + radius * np.sin(theta)
                    plt.plot(x, y, 'b-', linewidth=2)
                    
                    # Mark the trust center
                    plt.plot(center[0], center[1], 'bx', markersize=8)
                except Exception as e:
                    plt.figtext(0.5, 0.01, f"Error plotting trust region: {e}", 
                                ha='center', color='red')
            
            plt.show()
        
        with info_output:
            info_output.clear_output(wait=True)
            # Print current point info
            curr = result['current_point']
            print(f"Initial point: x1 = {curr[0]:.6f}, x2 = {curr[1]:.6f}")
            print(f"Objective function value: {curr[2]:.6f}")
            print(f"Constraint 1 value: {curr[3]:.6f} {'(Feasible)' if curr[5] else '(Infeasible)'}")
            print(f"Constraint 2 value: {curr[4]:.6f} {'(Feasible)' if curr[6] else '(Infeasible)'}")
            
            # Print all points info if there are any
            if points and point_values:
                print("\nIteration Progress:")
                for i, (point, value) in enumerate(zip(points, point_values)):
                    constraint1_val = -2.25*point[0] + point[1]
                    constraint2_val = point[0] + 1.5*point[1]
                    is_feasible1 = constraint1_val <= c1
                    is_feasible2 = constraint2_val <= c2
                    status = "FEASIBLE" if (is_feasible1 and is_feasible2) else "INFEASIBLE"
                    print(f"Iteration {i+1}: ({point[0]:.4f}, {point[1]:.4f}) | f={value:.6f} | c1={constraint1_val:.6f}, c2={constraint2_val:.6f} | {status}")
            
            print("\nReference:")
            print("Constrained global minimum is at (0.17143, -0.11429) with value 2.750737")

    def on_add_clicked(b):
        nonlocal points, point_values
        new_point = [x1_slider.value, x2_slider.value]
        points.append(new_point)
        point_values.append(rosenbrock(new_point))
        update_plot(x1_slider.value, x2_slider.value)
    
    def on_plot_points_clicked(b):
        nonlocal points, point_values
        try:
            # Parse the text input to get points
            text_points = points_input.value.strip().split('\n')
            new_points = []
            new_values = []
            
            for point_text in text_points:
                if point_text.strip():
                    coords = list(map(float, point_text.split(',')))
                    if len(coords) == 2:
                        new_points.append(coords)
                        new_values.append(rosenbrock(coords))
            
            points = new_points
            point_values = new_values
            update_plot(x1_slider.value, x2_slider.value)
        except Exception as e:
            with info_output:
                info_output.clear_output(wait=True)
                print(f"Error parsing points: {e}")
                print("Make sure each line contains a comma-separated pair of numbers.")
    
    def on_clear_clicked(b):
        nonlocal points, point_values
        points = []
        point_values = []
        points_input.value = ''
        update_plot(x1_slider.value, x2_slider.value)
    
    add_button.on_click(on_add_clicked)
    plot_points_button.on_click(on_plot_points_clicked)
    clear_button.on_click(on_clear_clicked)
    
     # Display widgets first, then update the plot
    display(widgets.HBox([x1_slider, x2_slider]))
    display(points_input)
    display(widgets.HBox([add_button, plot_points_button, clear_button, show_annotations_checkbox]))
    display(widgets.HBox([trust_center_input, trust_radius_input, show_trust_checkbox]))
    display(plot_output)
    display(info_output)
    
    # Update plot when sliders change
    def on_value_change(change):
        update_plot(x1_slider.value, x2_slider.value)
    
    x1_slider.observe(on_value_change, names='value')
    x2_slider.observe(on_value_change, names='value')
    show_annotations_checkbox.observe(lambda change: update_plot(x1_slider.value, x2_slider.value), names='value')
    
    # Update plot when trust region settings change
    trust_center_input.observe(lambda change: update_plot(x1_slider.value, x2_slider.value), names='value')
    trust_radius_input.observe(lambda change: update_plot(x1_slider.value, x2_slider.value), names='value')
    show_trust_checkbox.observe(lambda change: update_plot(x1_slider.value, x2_slider.value), names='value')
    
    # Initial plot - after all widgets are displayed
    update_plot(x1_slider.value, x2_slider.value)

# Run the interactive visualization
interactive_rosenbrock()

Textarea(value='', description='Points:', layout=Layout(height='100px', width='300px'), placeholder='Enter poi…

Output()

Output()

In [7]:
def rosenbrock_gradient_fd(x, h=1e-5):
    """
    Compute the gradient of the Rosenbrock function using forward difference method.
    
    Parameters:
    x (array-like): Point [x1, x2] at which to evaluate the gradient
    h (float): Step size for finite difference
    
    Returns:
    numpy.ndarray: Gradient vector [df/dx1, df/dx2]
    """
    grad = np.zeros_like(x)
    f_x = rosenbrock(x)
    
    # Compute partial derivatives using forward difference
    for i in range(len(x)):
        x_perturbed = x.copy()
        x_perturbed[i] += abs(x_perturbed[i]) * h
        f_perturbed = rosenbrock(x_perturbed)
        grad[i] = (f_perturbed - f_x) / ((x_perturbed[i] - x[i]))
    
    return grad

def rosenbrock_gradient(x):
    x1, x2 = x[0], x[1]
    df_dx1 = -400 * x1 * (x2 - x1**2) - 2 * (1 - x1)
    df_dx2 = 200 * (x2 - x1**2)
    return np.array([df_dx1, df_dx2])

# Example usage
test_points = [
    [0.1, -1.6],
    [0.10, -0.6614],
    [0.872364, 0.45126],
    [0.9977, 0.69604],
    [1.1214, 0.9721],
    [0.971629, 0.675],
    [1.0, 1.0]
]

print("Rosenbrock function gradient using forward difference:")
for point in test_points:
    x = np.array(point)
    fd_gradient = rosenbrock_gradient_fd(x)
    analytical_gradient = rosenbrock_gradient(x)
    function_value = rosenbrock(x)
    
    print(f"Point: {point}")
    print(f"Function value: {function_value:.6f}")
    print(f"Forward difference gradient: [{fd_gradient[0]:.6f}, {fd_gradient[1]:.6f}]")
    print(f"Analytical gradient: [{analytical_gradient[0]:.6f}, {analytical_gradient[1]:.6f}]")
    print("-" * 40)

Rosenbrock function gradient using forward difference:
Point: [0.1, -1.6]
Function value: 260.020000
Forward difference gradient: [62.600327, -321.998400]
Analytical gradient: [62.600000, -322.000000]
----------------------------------------
Point: [0.1, -0.6614]
Function value: 45.887796
Forward difference gradient: [25.056139, -134.279339]
Analytical gradient: [25.056000, -134.280000]
----------------------------------------
Point: [0.872364, 0.45126]
Function value: 9.611352
Forward difference gradient: [107.836955, -61.951338]
Analytical gradient: [107.833750, -61.951790]
----------------------------------------
Point: [0.9977, 0.69604]
Function value: 8.961963
Forward difference gradient: [119.470680, -59.872362]
Analytical gradient: [119.466100, -59.873058]
----------------------------------------
Point: [1.1214, 0.9721]
Function value: 8.162221
Forward difference gradient: [128.285144, -57.086620]
Analytical gradient: [128.278851, -57.087592]
------------------------------------